# 03 · Modelado, optimización, calibración y evaluación final
Protocolo temporal de la Fase 2 (v5, 6.2):
1. **Partición final:** se repite la regla de C1 (auditoría de la Fase 1). Con datos reales, C1 queda excluida y se entrena con **C2+C3**.
2. **Optimización con Optuna** dentro del entrenamiento (C2 → C3) para dos modelos obligatorios: **regresión logística regularizada** y **gradient boosting** con monotonía en atrasos.
3. **Selección en C4** frente a las reglas sin IA (**D** y **D2**), con la métrica principal: Lift@k y Precision@k por representante.
4. **Calibración** (Platt) en C4, explicación con **SHAP** y **congelamiento** del sistema (huella SHA-256).
5. **Prueba final en C5, una sola vez:** IC 95 % por bootstrap, Brier frente a R/B0/B1, equidad y proyección por sede y subnivel.

Con datos reales solo se descargan **agregados** (celdas con menos de 5 casos ocultas).

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML',
              'optuna': 'optuna', 'shap': 'shap', 'seaborn': 'seaborn'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title Cargar la base
FUENTE = "sintetica"  # @param ["sintetica", "real"]
# Marque SUBIR_DE_NUEVO si subió un archivo equivocado (se borra y se vuelve a pedir)
SUBIR_DE_NUEVO = False  # @param {type:"boolean"}

import pandas as pd
from src.data_processing import construir_dataset, limpiar_base
from src.utils import cargar_config, configurar_logging, obtener_base_seud

configurar_logging()
config = cargar_config()
MIN = config["privacidad"]["min_celda"]
ruta = RAIZ / config["rutas"]["base_real" if FUENTE == "real" else "base_sintetica"]

crudo = (obtener_base_seud(ruta, EN_COLAB, SUBIR_DE_NUEVO) if FUENTE == "real"
         else pd.read_csv(ruta, dtype=str))
base = limpiar_base(crudo, config)
dataset, reporte = construir_dataset(base, config)
if dataset.empty or "y_no_matricula" not in dataset:
    anios = crudo["anoa"].astype(str).str.strip().value_counts().head(10).to_dict()
    raise RuntimeError(
        "No se pudo construir ninguna cohorte. Revise la columna anoa: debe tener el año "
        f"de inicio del ciclo (2021…2026). Valores más frecuentes: {anios}")
print(f"Fuente: {FUENTE} | filas base: {len(base)} | filas modelables: {len(dataset)}")
if FUENTE == "sintetica":
    print("⚠️ Datos SINTÉTICOS: sirven para probar el código, no describen a LEMAS.")

## 1. Partición final y capacidad k
Se repite la regla registrada de la Fase 1 (tolerancia 0,02 en Precision@k de C4) para fijar las cohortes de entrenamiento. **C5 no interviene.**

In [ ]:
from src.auditoria import comparar_configuraciones, conteo_eventos, decidir_inclusion_c1
from src.evaluate import calcular_k_por_sede, consolidar_familias, verificar_capacidad
import numpy as np

sin_prueba = dataset[dataset["rol"] != "prueba"]
comparacion = comparar_configuraciones(sin_prueba, config)
INCLUIR_C1, motivo = decidir_inclusion_c1(comparacion, conteo_eventos(dataset, config), config)
COHORTES_ENTRENAMIENTO = config["auditoria"]["configuraciones_entrenamiento"][
    "con_C1" if INCLUIR_C1 else "sin_C1"]
print(("✅ " if INCLUIR_C1 else "⚠️ ") + motivo)
print("Entrenamiento:", COHORTES_ENTRENAMIENTO, "| Selección: C4 | Prueba: C5")

entrenamiento = dataset[dataset["cohorte"].isin(COHORTES_ENTRENAMIENTO)]
seleccion = dataset[dataset["rol"] == "seleccion"]
prueba = dataset[dataset["rol"] == "prueba"]
K = calcular_k_por_sede(consolidar_familias(entrenamiento, np.zeros(len(entrenamiento))),
                        config["capacidad"].get("multiplicador_k", 1.0))
print("k por sede:", K)
display(verificar_capacidad(K, config))

## 2. Optimización de hiperparámetros (Optuna)
Validación interna temporal: se entrena con la cohorte más antigua del entrenamiento y se valida con la más reciente. El objetivo es la **PR-AUC**, más estable que Precision@k con pocos eventos. Cada candidato se reentrena luego con todo el entrenamiento.

In [ ]:
# @title Candidatos
N_TRIALS = 60  # @param {type:"integer"}
from src.modeling import NOMBRES, diagnostico_ajuste, entrenar_candidatos, tabla_seleccion

SEMILLA = config["proyecto"]["semilla"]
candidatos = entrenar_candidatos(dataset, config, COHORTES_ENTRENAMIENTO, K, n_trials=N_TRIALS)
for c in candidatos.values():
    print(f"{NOMBRES[c.tipo]} | PR-AUC validación interna = {c.pr_auc_validacion_interna}")
    print("   hiperparámetros:", {k: (round(v, 4) if isinstance(v, float) else v)
                                 for k, v in c.params.items()})

## 3. Selección en C4 y diagnóstico de ajuste
Se ordena por **Lift@k** (desempate: PR-AUC). El modelo debe superar a la mejor regla (**D** o **D2**); si no, la recomendación operativa es esa regla. El diagnóstico compara la PR-AUC en entrenamiento y en C4 (actividad de la semana 3).

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
FIG = RAIZ / config["rutas"]["figuras"]; FIG.mkdir(parents=True, exist_ok=True)
MET = RAIZ / config["rutas"]["metricas"]; MET.mkdir(parents=True, exist_ok=True)
DPI = config["figuras"]["dpi"]

seleccion_c4 = tabla_seleccion(candidatos, dataset, K, SEMILLA)
display(seleccion_c4)
ajuste = diagnostico_ajuste(candidatos)
display(ajuste)

mejor_ml = max(candidatos.values(), key=lambda c: (c.metricas_c4["lift_k"], c.metricas_c4["pr_auc"]))
reglas = seleccion_c4[seleccion_c4["tipo"] == "regla"]
mejor_regla = reglas.iloc[0]          # la tabla ya está ordenada por Lift@k
MODELO_SUPERA_REGLAS = mejor_ml.metricas_c4["lift_k"] > mejor_regla["lift_k"]
PRIORIZACION = NOMBRES[mejor_ml.tipo] if MODELO_SUPERA_REGLAS else mejor_regla["candidato"]
print(f"Mejor modelo: {NOMBRES[mejor_ml.tipo]} (Lift@k C4 = {mejor_ml.metricas_c4['lift_k']:.2f})")
print(f"Mejor regla: {mejor_regla['candidato']} (Lift@k C4 = {mejor_regla['lift_k']:.2f})")
print(("✅ El modelo supera a las reglas D y D2: se usa para priorizar." if MODELO_SUPERA_REGLAS else
       f"⚠️ El modelo NO supera a la mejor regla: la priorización recomendada es {PRIORIZACION}; "
       "el modelo se conserva para probabilidades y proyección."))

fig, ax = plt.subplots(figsize=(9, 4))
orden = seleccion_c4.sort_values("lift_k")
ax.barh(orden["candidato"], orden["lift_k"],
        color=["#8B1E3F" if t != "regla" else "#9DB4C9" for t in orden["tipo"]])
ax.axvline(1, ls="--", c="gray")
ax.set(title="Figura 9. Lift@k por representante en C4 (selección)", xlabel="Lift@k (1 = azar)")
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_09_seleccion_c4.png", dpi=DPI, bbox_inches="tight")
plt.show()

seleccion_c4.to_csv(MET / f"seleccion_C4_{FUENTE}.csv", index=False)
ajuste.to_csv(MET / f"diagnostico_ajuste_{FUENTE}.csv", index=False)

## 4. Calibración en C4
Calibración de Platt sobre el modelo congelado. **No cambia el orden** (ni Precision@k), solo hace que el riesgo se lea como probabilidad. Como se ajusta en C4, la curva en C4 es optimista: el Brier válido es el de C5.

In [ ]:
from sklearn.calibration import calibration_curve
from src.data_processing import construir_matriz_x
from src.modeling import calibrar

calibrado = calibrar(mejor_ml.modelo, seleccion)
X4, y4 = construir_matriz_x(seleccion), seleccion["y_no_matricula"].to_numpy()
fig, ax = plt.subplots(figsize=(5.5, 5))
for nombre, prob in [("sin calibrar", mejor_ml.modelo.predict_proba(X4)[:, 1]),
                     ("calibrado (Platt)", calibrado.predict_proba(X4)[:, 1])]:
    frac, media = calibration_curve(y4, prob, n_bins=8, strategy="quantile")
    ax.plot(media, frac, marker="o", label=nombre)
ax.plot([0, 1], [0, 1], ls="--", c="gray")
ax.set(title="Figura 10. Curva de calibración en C4", xlabel="riesgo estimado",
       ylabel="no matrícula observada", xlim=(0, 0.6), ylim=(0, 0.6))
ax.legend()
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_10_calibracion.png", dpi=DPI, bbox_inches="tight")
plt.show()

## 5. Explicabilidad (SHAP y permutación)
Con datos reales solo se muestra la **importancia media** (agregada). El gráfico por estudiante (beeswarm) se muestra únicamente con datos sintéticos.

In [ ]:
import shap
from sklearn.inspection import permutation_importance

prep, clf = mejor_ml.modelo.named_steps["prep"], mejor_ml.modelo.named_steps["modelo"]
Xt = prep.transform(X4)
Xt = pd.DataFrame(Xt, columns=prep.get_feature_names_out()) if not hasattr(Xt, "columns") else Xt
if mejor_ml.tipo == "arboles":
    explicador = shap.TreeExplainer(clf)
    valores = explicador(Xt)
else:
    fondo = prep.transform(construir_matriz_x(entrenamiento))
    explicador = shap.LinearExplainer(clf, fondo)
    valores = explicador(Xt)
if valores.values.ndim == 3:   # algunos explicadores devuelven ambas clases
    valores = valores[:, :, 1]
importancia = (pd.DataFrame({"variable": Xt.columns, "shap_medio_abs": np.abs(valores.values).mean(0)})
               .sort_values("shap_medio_abs", ascending=False).reset_index(drop=True))
display(importancia.head(15))
importancia.to_csv(MET / f"shap_importancia_{FUENTE}.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
top = importancia.head(12).iloc[::-1]
ax.barh(top["variable"], top["shap_medio_abs"], color="#8B1E3F")
ax.set(title=f"Figura 11. Importancia SHAP media · {NOMBRES[mejor_ml.tipo]} (C4)",
       xlabel="|SHAP| medio (efecto en el riesgo)")
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_11_shap_importancia.png", dpi=DPI, bbox_inches="tight")
plt.show()
if FUENTE == "sintetica":
    shap.plots.beeswarm(valores, max_display=12, show=False)
    plt.title("SHAP por estudiante (solo datos sintéticos)"); plt.show()

perm = permutation_importance(mejor_ml.modelo, X4, y4, scoring="average_precision",
                              n_repeats=30, random_state=SEMILLA)
permutacion = (pd.DataFrame({"variable": X4.columns, "caida_pr_auc": perm.importances_mean,
                             "desv": perm.importances_std})
               .sort_values("caida_pr_auc", ascending=False).round(4))
display(permutacion)
permutacion.to_csv(MET / f"permutacion_{FUENTE}.csv", index=False)

## 6. Congelar el sistema
Se guarda todo lo necesario para la prueba final y para la app (modelo, calibración, k, cohortes, hiperparámetros y métricas de C4). La huella SHA-256 se imprime **antes** de abrir C5.

In [ ]:
import datetime
from src.modeling import guardar_sistema

sistema = {
    "fuente": FUENTE, "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "modelo": mejor_ml.modelo, "calibrado": calibrado, "tipo": mejor_ml.tipo,
    "nombre": NOMBRES[mejor_ml.tipo], "hiperparametros": mejor_ml.params,
    "cohortes_entrenamiento": COHORTES_ENTRENAMIENTO, "incluir_C1": bool(INCLUIR_C1),
    "k_por_sede": K, "priorizacion": PRIORIZACION, "modelo_supera_reglas": bool(MODELO_SUPERA_REGLAS),
    "metricas_C4": mejor_ml.metricas_c4,
    "tasa_historica_no_matricula": float(entrenamiento["y_no_matricula"].mean()),
}
RUTA_SISTEMA = RAIZ / config["rutas"]["modelos"] / f"sistema_{FUENTE}.joblib"
HUELLA = guardar_sistema(sistema, RUTA_SISTEMA)
print("Sistema congelado:", RUTA_SISTEMA.name, "| SHA-256:", HUELLA)

## 7. Prueba final en C5 (una sola vez)
Marque `EVALUAR_C5` solo cuando la selección en C4 esté revisada. Desde aquí **no se cambia nada** del sistema: si se modifica algo, la evaluación en C5 deja de ser una prueba independiente y debe declararse.

In [ ]:
# @title Confirmar la evaluación final
EVALUAR_C5 = False  # @param {type:"boolean"}
if not EVALUAR_C5:
    raise SystemExit("C5 no evaluada. Marque EVALUAR_C5 para la prueba final.")

from src.modeling import bootstrap_ic, equidad, lineas_base_brier, metricas, proyeccion, puntajes_reglas
X5 = construir_matriz_x(prueba)
riesgo5 = calibrado.predict_proba(X5)[:, 1]
filas, intervalos = [], {}
for nombre, puntaje in [(NOMBRES[mejor_ml.tipo], riesgo5), *puntajes_reglas(prueba).items()]:
    filas.append({"sistema": nombre, **metricas(prueba, puntaje, K, SEMILLA)})
    intervalos[nombre] = bootstrap_ic(prueba, puntaje, K, SEMILLA, repeticiones=2000)
rng = np.random.default_rng(SEMILLA)
azar = pd.DataFrame([metricas(prueba, rng.random(len(prueba)), K, SEMILLA) for _ in range(300)]).drop(columns="brier")
filas.append({"sistema": "A · selección al azar de k familias", **azar.mean().round(4).to_dict()})
final = pd.DataFrame(filas)
for col in ("precision_k", "lift_k", "recall_k"):
    final[f"ic95_{col}"] = final["sistema"].map(lambda s: intervalos.get(s, {}).get(col))
display(final)
brier = pd.concat([lineas_base_brier(prueba, entrenamiento),
                   pd.DataFrame([{"linea_base": f"Modelo calibrado · {NOMBRES[mejor_ml.tipo]}",
                                  "brier": round(final.loc[0, "brier"], 4)}])], ignore_index=True)
display(brier)

In [ ]:
# Equidad (por representante, top-k) y proyección por sede y subnivel
GRUPOS = ["sede", "beca", "subnivel", "representante_atipico"]
riesgo_prior = riesgo5 if MODELO_SUPERA_REGLAS else puntajes_reglas(prueba)[PRIORIZACION]
tabla_equidad = equidad(prueba, riesgo_prior, K, SEMILLA, GRUPOS, minimo=MIN)
display(tabla_equidad)
tabla_proy = proyeccion(prueba, riesgo5, sistema["tasa_historica_no_matricula"])
display(tabla_proy)
mape = {"modelo": round(float(tabla_proy["ape_modelo"].mean()), 4),
        "B1": round(float(tabla_proy["ape_B1"].mean()), 4)}
print("MAPE por sede y subnivel (grupos con ≥ 10 matrículas):", mape)

fig, ax = plt.subplots(figsize=(9, 4))
vista = final.set_index("sistema")
err = np.array([[v - (vista.loc[s, "ic95_lift_k"] or [v, v])[0], (vista.loc[s, "ic95_lift_k"] or [v, v])[1] - v]
                for s, v in vista["lift_k"].items()]).T
ax.barh(vista.index, vista["lift_k"], xerr=err, color="#8B1E3F", capsize=4)
ax.axvline(1, ls="--", c="gray")
ax.set(title="Figura 12. Lift@k en C5 (prueba final) con IC 95 %", xlabel="Lift@k (1 = azar)")
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_12_prueba_c5.png", dpi=DPI, bbox_inches="tight")
plt.show()

resultado = {"fuente": FUENTE, "sha256_sistema": HUELLA, "k_por_sede": K,
             "cohortes_entrenamiento": COHORTES_ENTRENAMIENTO, "priorizacion": PRIORIZACION,
             "C4": seleccion_c4.to_dict(orient="records"),
             "C5": final.to_dict(orient="records"), "brier_C5": brier.to_dict(orient="records"),
             "mape_C5": mape, "hiperparametros": {c.tipo: c.params for c in candidatos.values()}}
(MET / f"resultados_finales_{FUENTE}.json").write_text(
    json.dumps(resultado, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
tabla_equidad.to_csv(MET / f"equidad_C5_{FUENTE}.csv", index=False)
tabla_proy.to_csv(MET / f"proyeccion_C5_{FUENTE}.csv", index=False)
print("Guardado en", MET)

In [ ]:
# @title Descargar agregados para compartir (Colab)
# Solo métricas y figuras agregadas. El modelo real (models/*_real*) NO se comparte.
import shutil
paquete = RAIZ / f"fase2_{FUENTE}"
shutil.rmtree(paquete, ignore_errors=True); paquete.mkdir()
for patron in [f"*_{FUENTE}.csv", f"*_{FUENTE}.json"]:
    for archivo in MET.glob(patron):
        if archivo.name.split("_")[0] in {"seleccion", "diagnostico", "shap", "permutacion",
                                           "resultados", "equidad", "proyeccion"}:
            shutil.copy(archivo, paquete)
for png in FIG.glob(f"{FUENTE}_1[0-2]_*.png"):
    shutil.copy(png, paquete)
for png in FIG.glob(f"{FUENTE}_09_*.png"):
    shutil.copy(png, paquete)
zip_final = shutil.make_archive(str(paquete), "zip", paquete)
print("Contenido:", sorted(p.name for p in paquete.iterdir()))
if EN_COLAB:
    from google.colab import files
    files.download(zip_final)